# The Multi-Model Dashboard: Streaming GPT and DeepSeek Side-by-Side

In the previous labs, we built the individual components of a frontier AI system. Today, we perform the **System Integration**. 

We are building a **Dual-Pane Dashboard**. This isn't just a UI; it's a benchmarking tool. We will implement interleaved streaming logic that allows us to watch two different "Brains" (OpenAI and DeepSeek) compete in real-time within the same visual interface. This allows for instant evaluation of accuracy, tone, and reasoning depth.

In [1]:
import gradio as gr
from litellm import completion
from dotenv import load_dotenv
import os
from itertools import zip_longest

load_dotenv(override=True)

print("🛰️ Multi-Model Command Center Initialized!")

🛰️ Multi-Model Command Center Initialized!


## 1. The Interleaved Streamer

To make the UI feel "simultaneous," we cannot wait for one model to finish before starting the other. We must **interleave** the chunks. We use `itertools.zip_longest` to pull one token from OpenAI, then one from DeepSeek, repeating until both are complete.

In [2]:
def dual_stream_battle(prompt):
    """
    Orchestrates two simultaneous streams and yields updates for both UI panes.
    """
    # 1. Start both streams
    stream_openai = completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    
    stream_deepseek = completion(
        model="deepseek/deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    text_openai = ""
    text_deepseek = ""

    # 2. Interleave the chunks for simultaneous-feeling UI updates
    for chunk_o, chunk_d in zip_longest(stream_openai, stream_deepseek):
        if chunk_o:
            content_o = chunk_o.choices[0].delta.content
            if content_o: text_openai += content_o
            
        if chunk_d:
            content_d = chunk_d.choices[0].delta.content
            if content_d: text_deepseek += content_d
            
        # 3. Yield the current state of BOTH textboxes
        yield text_openai, text_deepseek

## 2. Wiring the Dual-Pane Dashboard

We use `gr.Blocks` to create our layout. By mapping our generator's output to two different `Markdown` components, we create the "Battlefield" effect.

In [3]:
with gr.Blocks(theme=gr.themes.Default(primary_hue="orange")) as dashboard:
    gr.Markdown("# 🏆 Multi-Model Battle Dashboard")
    gr.Markdown("Enter a complex prompt to see how OpenAI and DeepSeek handle it side-by-side.")
    
    with gr.Row():
        user_prompt = gr.Textbox(
            label="The Challenge", 
            placeholder="E.g., Compare the economic impact of AI in 2025 vs 2030...",
            lines=2
        )
    
    battle_btn = gr.Button("🚀 Launch Dual Stream", variant="primary")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🤖 OpenAI GPT-4o-mini")
            openai_display = gr.Markdown(label="GPT Output")
            
        with gr.Column():
            gr.Markdown("### 🤖 DeepSeek-V3")
            deepseek_display = gr.Markdown(label="DeepSeek Output")

    # WIRING: Note how one function updates TWO outputs simultaneously
    battle_btn.click(
        fn=dual_stream_battle, 
        inputs=user_prompt, 
        outputs=[openai_display, deepseek_display]
    )

print("🚀 Dashboard Ready to Launch.")
dashboard.launch()

/var/folders/rg/b21vyw1s0v727y0fc2phllm00000gn/T/ipykernel_36167/1335036637.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="orange")) as dashboard:


🚀 Dashboard Ready to Launch.
* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
